# Surveying Computations: Distances Between Consecutive Points

This notebook downloads a shared coordinate file from Google Drive and calculates horizontal distances between consecutive points. Students do **not** need to mount their own Google Drive.

Expected columns: `P,N,E,Z` (`P` = point ID, `N` = Northing, `E` = Easting, `Z` = elevation). The horizontal distance uses only Northing and Easting:

`D = sqrt((N2 - N1)^2 + (E2 - E1)^2)`

**Important:** the owner must set the file's General access to **Anyone with the link — Viewer**. The file is read-only for students.

## Step 1 — Download the shared file

The file ID below comes from the teacher's Google Drive sharing link. If the teacher changes the file, keep the same file ID where possible.

In [ ]:
import pandas as pd
import numpy as np
from io import BytesIO
import requests

# Shared Google Drive file ID (teacher's file)
FILE_ID = '1jpbOe3JbJJ7W7cHLWx9bLBNFUapHoZK-'
DOWNLOAD_URL = f'https://drive.google.com/uc?export=download&id={FILE_ID}'

response = requests.get(DOWNLOAD_URL, timeout=30)
response.raise_for_status()

# Read the shared CSV/TXT file. It must be comma-separated.
from io import StringIO
points = pd.read_csv(StringIO(response.content.decode('utf-8-sig')), sep=',')
points.head()

## Step 2 — Validate coordinate columns

Rows are processed in the order they appear in the file. Do not sort the data unless that is intended.

In [ ]:
point_col = 'P'
northing_col = 'N'
easting_col = 'E'
elevation_col = 'Z'  # retained for reference; not used in horizontal distance

required = [point_col, northing_col, easting_col]
missing = [c for c in required if c not in points.columns]
if missing:
    raise KeyError(f'Missing required columns: {missing}. Available columns: {list(points.columns)}')

work = points.copy()
work[point_col] = work[point_col].astype(str).str.strip()
work[northing_col] = pd.to_numeric(work[northing_col], errors='coerce')
work[easting_col] = pd.to_numeric(work[easting_col], errors='coerce')
work = work.dropna(subset=[point_col, northing_col, easting_col]).reset_index(drop=True)

if len(work) < 2:
    raise ValueError('At least two valid points are required to calculate distances.')

display(work)

## Step 3 — Calculate horizontal distances

The calculation uses Northing and Easting only; elevation differences are not included.

In [ ]:
N1 = work[northing_col].iloc[:-1].to_numpy(dtype=float)
N2 = work[northing_col].iloc[1:].to_numpy(dtype=float)
E1 = work[easting_col].iloc[:-1].to_numpy(dtype=float)
E2 = work[easting_col].iloc[1:].to_numpy(dtype=float)

results = pd.DataFrame({
    'From_Point': work[point_col].iloc[:-1].to_numpy(),
    'To_Point': work[point_col].iloc[1:].to_numpy(),
    'Delta_N': N2 - N1,
    'Delta_E': E2 - E1,
    'Horizontal_Distance': np.sqrt((N2 - N1)**2 + (E2 - E1)**2)
})

display(results.round(3))
print(f"Number of point-to-point distances: {len(results)}")
print(f"Total consecutive horizontal distance: {results['Horizontal_Distance'].sum():.3f}")

## Step 4 — Export the results

The output CSV is downloaded to the student's computer.

In [ ]:
output_path = '/content/Consecutive_Point_Distances.csv'
results.to_csv(output_path, index=False, encoding='utf-8-sig')
print(f'Results saved to {output_path}')

from google.colab import files
files.download(output_path)